In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#
import os
import sys
from pathlib import Path

# غيّرها إلى True لو عايز تحفظ النتائج على Google Drive.
USE_GOOGLE_DRIVE = False

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/AQFE_QCVAE_Results")
else:
    # Works in Colab and local/Jupyter environments.
    DEFAULT_ROOT = Path("/content") if IN_COLAB else Path.cwd()
    BASE_DIR = DEFAULT_ROOT / "AQFE_QCVAE_Results"

DATA_DIR = BASE_DIR / "data"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
FIGURE_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"

for p in [BASE_DIR, DATA_DIR, CHECKPOINT_DIR, FIGURE_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("FIGURE_DIR:", FIGURE_DIR)
print("TABLE_DIR:", TABLE_DIR)


In [ ]:
# 

import math
import json
import random
import time
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from contextlib import nullcontext

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")


In [ ]:
#============================================================

@dataclass
class Config:
    # تشغيل سريع للتأكد أو تشغيل كامل للبحث
    mode: str = "paper"  # "debug" أو "paper"

    # الداتا سيتس المطلوبة
    datasets: Tuple[str, ...] = ("MNIST", "FashionMNIST")

    # لو عايز mean ± std حقيقي عبر أكثر من training run، استخدم أكثر من seed.
    # مثال للبحث: (42, 123, 2026)
    seeds: Tuple[int, ...] = (42,)

    # training parameters
    epochs: int = 80
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    use_amp: bool = True
    num_workers: int = 2

    # model parameters
    image_size: int = 28
    patch_size: int = 3
    stride: int = 1
    aqfe_embed_dim: int = 16
    q_layers: int = 3
    theta_scale: float = math.pi
    latent_dim: int = 32
    hidden_dim: int = 256

    # VAE / loss parameters
    beta_max: float = 0.05
    kl_warmup_epochs: int = 20
    ssim_weight: float = 0.15
    feature_weight: float = 0.03
    sparsity_weight: float = 0.002
    circuit_weight: float = 1e-4
    keep_ratio: float = 0.45

    # saving / loading behavior
    force_retrain: bool = False
    resume_if_available: bool = True
    save_every_epochs: int = 10
    visual_every_epochs: int = 10

    # debug mode sizes
    debug_train_subset: int = 1024
    debug_test_subset: int = 256

    # paper mode sizes: None يعني استخدم كل الداتا
    paper_train_subset: Optional[int] = None
    paper_test_subset: Optional[int] = None

    # evaluation/statistics parameters
    run_statistics: bool = True
    eval_max_batches: Optional[int] = None
    eval_batch_size: int = 128
    fid_image_size: int = 299
    lpips_image_size: int = 64
    fid_chunk_size: int = 256
    noisy_depolarizing_p: float = 0.03

CFG = Config()

if CFG.mode == "debug":
    CFG.epochs = 1
    CFG.seeds = (42,)
    CFG.eval_max_batches = 2
    TRAIN_SUBSET_SIZE = CFG.debug_train_subset
    TEST_SUBSET_SIZE = CFG.debug_test_subset
elif CFG.mode == "paper":
    CFG.epochs = 80
    CFG.eval_max_batches = None
    TRAIN_SUBSET_SIZE = CFG.paper_train_subset
    TEST_SUBSET_SIZE = CFG.paper_test_subset
else:
    raise ValueError('CFG.mode must be either "debug" or "paper"')

print(CFG)
print("TRAIN_SUBSET_SIZE:", TRAIN_SUBSET_SIZE)
print("TEST_SUBSET_SIZE:", TEST_SUBSET_SIZE)


In [ ]:
# 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"


def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def autocast_context(enabled: bool = True):
    """AMP autocast context that avoids deprecated torch.cuda.amp.autocast."""
    if DEVICE.type == "cuda":
        try:
            return torch.amp.autocast(device_type="cuda", enabled=enabled)
        except TypeError:
            return torch.cuda.amp.autocast(enabled=enabled)
    return nullcontext()


def make_grad_scaler(enabled: bool = True):
    """Create GradScaler with compatibility fallback."""
    use_scaler = bool(enabled and DEVICE.type == "cuda")
    try:
        return torch.amp.GradScaler("cuda", enabled=use_scaler)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=use_scaler)

print("DEVICE:", DEVICE)
print("PIN_MEMORY:", PIN_MEMORY)


In [ ]:

DATASET_REGISTRY = {
    "MNIST": {
        "class": datasets.MNIST,
        "classes": [str(i) for i in range(10)],
    },
    "FashionMNIST": {
        "class": datasets.FashionMNIST,
        "classes": [
            "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
            "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
        ],
    },
}


def make_subset(dataset, subset_size: Optional[int], seed: int):
    """Return a deterministic subset if subset_size is set."""
    if subset_size is None or subset_size >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:subset_size].tolist()
    return Subset(dataset, indices)


def build_dataloaders(dataset_name: str, seed: int):
    """Build train and test loaders for MNIST or Fashion-MNIST."""
    if dataset_name not in DATASET_REGISTRY:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    dataset_cls = DATASET_REGISTRY[dataset_name]["class"]

    train_dataset = dataset_cls(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=transform,
    )

    test_dataset = dataset_cls(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=transform,
    )

    train_dataset = make_subset(train_dataset, TRAIN_SUBSET_SIZE, seed)
    test_dataset = make_subset(test_dataset, TEST_SUBSET_SIZE, seed + 999)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    return train_loader, test_loader, DATASET_REGISTRY[dataset_name]["classes"]

# quick visual sanity check for both datasets
for ds_name in CFG.datasets:
    loader, _, classes = build_dataloaders(ds_name, seed=42)
    x_preview, y_preview = next(iter(loader))
    print(ds_name, x_preview.shape, y_preview[:8].tolist())


In [ ]:
#

def show_tensor_grid(images: torch.Tensor, title: str = "", nrow: int = 8, save_path: Optional[Path] = None):
    """Show and optionally save a tensor image grid."""
    images = images.detach().cpu().float().clamp(0, 1)
    grid = make_grid(images, nrow=nrow, padding=2)

    plt.figure(figsize=(min(12, nrow * 1.4), 4))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_image(images, str(save_path), nrow=nrow)
        print("Saved:", save_path)


def show_original_recon_diff(original: torch.Tensor, recon: torch.Tensor, title: str = "", save_path: Optional[Path] = None):
    """Show original, reconstructed, and absolute difference for one image."""
    original = original.detach().cpu().float().clamp(0, 1)
    recon = recon.detach().cpu().float().clamp(0, 1)
    diff = (original - recon).abs().clamp(0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    items = [(original, "Input"), (recon, "Regenerated / Reconstructed"), (diff, "Absolute Difference")]

    for ax, (img, name) in zip(axes, items):
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(name)
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()


In [ ]:
# 

def simple_ssim_loss(pred: torch.Tensor, target: torch.Tensor, window_size: int = 7) -> torch.Tensor:
    """A simple differentiable SSIM loss for grayscale images in [0, 1]."""
    pred = pred.float().clamp(0, 1)
    target = target.float().clamp(0, 1)

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    padding = window_size // 2

    mu_x = F.avg_pool2d(pred, kernel_size=window_size, stride=1, padding=padding)
    mu_y = F.avg_pool2d(target, kernel_size=window_size, stride=1, padding=padding)

    sigma_x = F.avg_pool2d(pred * pred, window_size, 1, padding) - mu_x * mu_x
    sigma_y = F.avg_pool2d(target * target, window_size, 1, padding) - mu_y * mu_y
    sigma_xy = F.avg_pool2d(pred * target, window_size, 1, padding) - mu_x * mu_y

    numerator = (2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)
    denominator = (mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2)

    ssim_map = numerator / (denominator + 1e-8)
    ssim = ssim_map.mean().clamp(0, 1)
    return 1.0 - ssim


def kl_weight_for_epoch(epoch: int, cfg: Config) -> float:
    """Linear KL warmup up to beta_max."""
    if cfg.kl_warmup_epochs <= 0:
        return cfg.beta_max
    progress = min(1.0, float(epoch) / float(cfg.kl_warmup_epochs))
    return cfg.beta_max * progress


def mean_dict(dicts: List[Dict[str, float]]) -> Dict[str, float]:
    """Average a list of metric dictionaries."""
    if len(dicts) == 0:
        return {}
    keys = dicts[0].keys()
    return {k: float(np.mean([d[k] for d in dicts])) for k in keys}


def mse_psnr(pred: torch.Tensor, target: torch.Tensor) -> Tuple[float, float]:
    """Compute MSE and PSNR for one batch/image."""
    mse = F.mse_loss(pred.float(), target.float()).item()
    psnr = 10.0 * math.log10(1.0 / max(mse, 1e-12))
    return mse, psnr


In [ ]:
# 
class AQFEFeatureExtractor(nn.Module):
    """Extract stable handcrafted features from local image patches."""

    def __init__(self, patch_size: int = 3, stride: int = 1):
        super().__init__()
        if patch_size % 2 == 0:
            raise ValueError("patch_size should be odd, e.g., 3 or 5")
        self.patch_size = patch_size
        self.stride = stride
        self.padding = patch_size // 2
        self.feature_dim = 8

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Input:  x [B, 1, H, W]
        Output: features [B, H*W, 8] when stride=1 and padding=patch_size//2
        """
        x = x.float().clamp(0, 1)
        b, c, h, w = x.shape
        if c != 1:
            raise ValueError("This extractor expects grayscale images with one channel.")

        patches = F.unfold(
            x,
            kernel_size=self.patch_size,
            stride=self.stride,
            padding=self.padding,
        )
        patches = patches.transpose(1, 2)  # [B, N, K*K]

        k = self.patch_size
        patch_img = patches.contiguous().view(b, -1, k, k)

        mean = patches.mean(dim=-1)
        std = patches.std(dim=-1, unbiased=False)
        pmax = patches.max(dim=-1).values
        pmin = patches.min(dim=-1).values
        contrast = pmax - pmin

        center = patch_img[:, :, k // 2, k // 2]

        left = patch_img[:, :, :, 0].mean(dim=-1)
        right = patch_img[:, :, :, -1].mean(dim=-1)
        top = patch_img[:, :, 0, :].mean(dim=-1)
        bottom = patch_img[:, :, -1, :].mean(dim=-1)

        gx = right - left
        gy = bottom - top
        edge = torch.sqrt(gx * gx + gy * gy + 1e-8)

        p = mean.clamp(1e-6, 1.0 - 1e-6)
        entropy = -(p * torch.log(p) + (1 - p) * torch.log(1 - p))

        features = torch.stack([
            mean,
            std,
            gx,
            gy,
            edge,
            contrast,
            center,
            entropy,
        ], dim=-1)

        return features


In [ ]:
# 
class SingleQubitDataReuploading(nn.Module):
    """Differentiable single-qubit Bloch-vector simulation."""

    def __init__(self, feature_dim: int, q_layers: int = 3, theta_scale: float = math.pi):
        super().__init__()
        self.feature_dim = feature_dim
        self.q_layers = q_layers
        self.theta_scale = theta_scale
        self.depolarizing_p = 0.0

        self.angle_projector = nn.Linear(feature_dim, 3)
        self.layer_scale = nn.Parameter(torch.ones(q_layers, 3))
        self.layer_bias = nn.Parameter(torch.zeros(q_layers, 3))

    @staticmethod
    def rotate_x(x, y, z, angle):
        ca = torch.cos(angle)
        sa = torch.sin(angle)
        y_new = y * ca - z * sa
        z_new = y * sa + z * ca
        return x, y_new, z_new

    @staticmethod
    def rotate_y(x, y, z, angle):
        ca = torch.cos(angle)
        sa = torch.sin(angle)
        x_new = x * ca + z * sa
        z_new = -x * sa + z * ca
        return x_new, y, z_new

    @staticmethod
    def rotate_z(x, y, z, angle):
        ca = torch.cos(angle)
        sa = torch.sin(angle)
        x_new = x * ca - y * sa
        y_new = x * sa + y * ca
        return x_new, y_new, z

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        """
        Input:  features [B, N, feature_dim]
        Output: qubit readouts [B, N, 4] = <X>, <Y>, <Z>, P(|0>)
        """
        angles = self.angle_projector(features.float()) * self.theta_scale

        b, n, _ = angles.shape
        x = torch.zeros(b, n, device=features.device, dtype=features.dtype)
        y = torch.zeros_like(x)
        z = torch.ones_like(x)

        for layer_idx in range(self.q_layers):
            a = angles * self.layer_scale[layer_idx].view(1, 1, 3) + self.layer_bias[layer_idx].view(1, 1, 3)
            x, y, z = self.rotate_x(x, y, z, a[..., 0])
            x, y, z = self.rotate_y(x, y, z, a[..., 1])
            x, y, z = self.rotate_z(x, y, z, a[..., 2])

        # depolarizing noise shrinks the Bloch vector toward the mixed state
        if self.depolarizing_p > 0:
            factor = max(0.0, 1.0 - float(self.depolarizing_p))
            x = factor * x
            y = factor * y
            z = factor * z

        p0 = (z + 1.0) / 2.0
        return torch.stack([x, y, z, p0], dim=-1)

    def regularization_loss(self) -> torch.Tensor:
        """Small L2 regularization for quantum parameters."""
        return (self.layer_scale.pow(2).mean() + self.layer_bias.pow(2).mean())


In [ ]:
# 

class AQFEModule(nn.Module):
    """AQFE module that keeps spatial maps instead of destroying spatial information."""

    def __init__(self, image_size: int, patch_size: int, stride: int, embed_dim: int, q_layers: int, theta_scale: float):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.extractor = AQFEFeatureExtractor(patch_size=patch_size, stride=stride)
        self.quantum = SingleQubitDataReuploading(
            feature_dim=self.extractor.feature_dim,
            q_layers=q_layers,
            theta_scale=theta_scale,
        )

        combined_dim = self.extractor.feature_dim + 4

        self.importance_gate = nn.Sequential(
            nn.Linear(self.extractor.feature_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

        self.feature_projector = nn.Sequential(
            nn.Linear(combined_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, embed_dim),
            nn.ReLU(inplace=True),
        )

    def extract_handcrafted_features(self, x: torch.Tensor) -> torch.Tensor:
        """Expose handcrafted features for feature-preserving loss."""
        return self.extractor(x)

    def set_depolarizing_noise(self, p: float):
        """Set depolarizing probability inside the quantum layer."""
        self.quantum.depolarizing_p = float(p)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        features = self.extractor(x)
        scores = self.importance_gate(features)

        # Soft gate, not hard top-k. This keeps gradients and spatial details.
        gated_features = features * (0.5 + scores)

        q_readouts = self.quantum(gated_features)
        combined = torch.cat([gated_features, q_readouts], dim=-1)
        embedded = self.feature_projector(combined)  # [B, N, embed_dim]

        b = x.shape[0]
        h = self.image_size
        w = self.image_size
        fmap = embedded.transpose(1, 2).contiguous().view(b, self.embed_dim, h, w)

        aux = {
            "score_mean": scores.mean(),
            "score_std": scores.std(unbiased=False),
        }
        return fmap, aux


In [ ]:
# ============================================================
# AQFE-QCVAE model
# 

class AQFEQCVAE(nn.Module):
    """AQFE-QCVAE for 28x28 grayscale images."""

    def __init__(
        self,
        image_size: int = 28,
        patch_size: int = 3,
        stride: int = 1,
        aqfe_embed_dim: int = 16,
        q_layers: int = 3,
        theta_scale: float = math.pi,
        latent_dim: int = 32,
        hidden_dim: int = 256,
    ):
        super().__init__()
        self.image_size = image_size
        self.latent_dim = latent_dim

        self.aqfe = AQFEModule(
            image_size=image_size,
            patch_size=patch_size,
            stride=stride,
            embed_dim=aqfe_embed_dim,
            q_layers=q_layers,
            theta_scale=theta_scale,
        )

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(aqfe_embed_dim, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 14 -> 7
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        conv_out_dim = 128 * 7 * 7

        self.encoder_fc = nn.Sequential(
            nn.Linear(conv_out_dim, hidden_dim),
            nn.ReLU(inplace=True),
        )

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, conv_out_dim),
            nn.ReLU(inplace=True),
        )

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 7 -> 14
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 14 -> 28
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 1, kernel_size=3, padding=1),  # logits, no Sigmoid here
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
        fmap, aux = self.aqfe(x)
        h = self.encoder_conv(fmap)
        h = h.view(h.size(0), -1)
        h = self.encoder_fc(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h).clamp(-8.0, 8.0)
        return mu, logvar, aux

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = self.decoder_fc(z)
        h = h.view(z.size(0), 128, 7, 7)
        logits = self.decoder_conv(h)
        return logits

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
        mu, logvar, aux = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_logits = self.decode(z)
        return recon_logits, mu, logvar, aux

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor, deterministic: bool = True) -> torch.Tensor:
        """Return reconstructed probability image in [0, 1]."""
        was_training = self.training
        self.eval()
        mu, logvar, _ = self.encode(x)
        if deterministic:
            z = mu
        else:
            std = torch.exp(0.5 * logvar)
            z = mu + torch.randn_like(std) * std
        logits = self.decode(z)
        recon = torch.sigmoid(logits).clamp(0, 1)
        if was_training:
            self.train()
        return recon

    @torch.no_grad()
    def generate(self, n: int, device: torch.device) -> torch.Tensor:
        """Generate random samples from standard normal latent prior."""
        was_training = self.training
        self.eval()
        z = torch.randn(n, self.latent_dim, device=device)
        logits = self.decode(z)
        samples = torch.sigmoid(logits).clamp(0, 1)
        if was_training:
            self.train()
        return samples


def model_kwargs_from_cfg(cfg: Config) -> Dict[str, Any]:
    """Create model kwargs from config to save inside checkpoints."""
    return {
        "image_size": cfg.image_size,
        "patch_size": cfg.patch_size,
        "stride": cfg.stride,
        "aqfe_embed_dim": cfg.aqfe_embed_dim,
        "q_layers": cfg.q_layers,
        "theta_scale": cfg.theta_scale,
        "latent_dim": cfg.latent_dim,
        "hidden_dim": cfg.hidden_dim,
    }


def build_model(cfg: Config) -> AQFEQCVAE:
    model = AQFEQCVAE(**model_kwargs_from_cfg(cfg))
    return model.to(DEVICE)

# sanity check
set_seed(42)
model_sanity = build_model(CFG)
x_sanity = torch.randn(4, 1, 28, 28).to(DEVICE).clamp(0, 1)
with torch.no_grad():
    y_sanity, mu_sanity, logvar_sanity, aux_sanity = model_sanity(x_sanity)
print("Sanity output:", y_sanity.shape, mu_sanity.shape, logvar_sanity.shape, aux_sanity)
del model_sanity, x_sanity, y_sanity, mu_sanity, logvar_sanity
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# 
def compute_loss(
    model: AQFEQCVAE,
    x: torch.Tensor,
    recon_logits: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    aux: Dict[str, torch.Tensor],
    epoch: int,
    cfg: Config,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    x_float = x.float().clamp(0, 1)
    logits_float = recon_logits.float()
    recon_prob = torch.sigmoid(logits_float).clamp(0, 1)

    bce = F.binary_cross_entropy_with_logits(
        logits_float,
        x_float,
        reduction="mean",
    )

    # KL normalized by latent dimension for stable scale
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    beta = kl_weight_for_epoch(epoch, cfg)

    ssim = simple_ssim_loss(recon_prob, x_float)

    # feature preservation: target features detached, recon features differentiable
    with torch.no_grad():
        target_features = model.aqfe.extract_handcrafted_features(x_float).detach()
    recon_features = model.aqfe.extract_handcrafted_features(recon_prob)
    feature_loss = F.mse_loss(recon_features, target_features)

    score_mean = aux["score_mean"]
    sparsity_loss = (score_mean - cfg.keep_ratio).pow(2)

    circuit_loss = model.aqfe.quantum.regularization_loss()

    total = (
        bce
        + beta * kl
        + cfg.ssim_weight * ssim
        + cfg.feature_weight * feature_loss
        + cfg.sparsity_weight * sparsity_loss
        + cfg.circuit_weight * circuit_loss
    )

    mse, psnr = mse_psnr(recon_prob.detach(), x_float.detach())

    metrics = {
        "loss": float(total.detach().cpu()),
        "bce": float(bce.detach().cpu()),
        "kl": float(kl.detach().cpu()),
        "beta": float(beta),
        "ssim_loss": float(ssim.detach().cpu()),
        "feature_loss": float(feature_loss.detach().cpu()),
        "score_mean": float(score_mean.detach().cpu()),
        "mse": float(mse),
        "psnr": float(psnr),
    }

    return total, metrics


In [ ]:
# ============================================================
# Checkpoint saving and loading


def safe_name(dataset_name: str) -> str:
    return dataset_name.lower().replace("-", "_").replace(" ", "_")


def checkpoint_paths(dataset_name: str, seed: int) -> Dict[str, Path]:
    prefix = f"{safe_name(dataset_name)}_{CFG.mode}_ps{CFG.patch_size}_ql{CFG.q_layers}_ld{CFG.latent_dim}_seed{seed}"
    return {
        "best": CHECKPOINT_DIR / f"{prefix}_best.pt",
        "latest": CHECKPOINT_DIR / f"{prefix}_latest.pt",
        "history": TABLE_DIR / f"{prefix}_history.csv",
    }


def save_checkpoint(
    path: Path,
    model: AQFEQCVAE,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[Any],
    scaler: Optional[Any],
    cfg: Config,
    dataset_name: str,
    seed: int,
    epoch: int,
    history: List[Dict[str, float]],
    best_val_loss: float,
):
    path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "dataset_name": dataset_name,
        "seed": seed,
        "epoch": epoch,
        "model_kwargs": model_kwargs_from_cfg(cfg),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "cfg": asdict(cfg),
        "history": history,
        "best_val_loss": best_val_loss,
    }
    torch.save(checkpoint, path)


def load_checkpoint(path: Path, device: torch.device = DEVICE) -> Dict[str, Any]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def build_model_from_checkpoint(path: Path, device: torch.device = DEVICE) -> Tuple[AQFEQCVAE, Dict[str, Any]]:
    checkpoint = load_checkpoint(path, device=device)
    model = AQFEQCVAE(**checkpoint["model_kwargs"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)
    return model, checkpoint

print("Checkpoint functions are ready.")


In [ ]:
# ============================================================
# Training and validation functions
# ===========================================================


def train_one_epoch(model, loader, optimizer, scaler, epoch: int, cfg: Config):
    model.train()
    model.aqfe.set_depolarizing_noise(0.0)
    metric_list = []

    for batch in loader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        x = x.to(DEVICE, non_blocking=True).float().clamp(0, 1)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context(cfg.use_amp):
            recon_logits, mu, logvar, aux = model(x)
            loss, metrics = compute_loss(model, x, recon_logits, mu, logvar, aux, epoch, cfg)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        metric_list.append(metrics)

    return mean_dict(metric_list)


@torch.no_grad()
def validate_one_epoch(model, loader, epoch: int, cfg: Config):
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)
    metric_list = []

    for batch in loader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        x = x.to(DEVICE, non_blocking=True).float().clamp(0, 1)

        with autocast_context(cfg.use_amp):
            recon_logits, mu, logvar, aux = model(x)
            loss, metrics = compute_loss(model, x, recon_logits, mu, logvar, aux, epoch, cfg)

        metric_list.append(metrics)

    return mean_dict(metric_list)


def _display_tensor_grid(images, title: str, nrow: int = 8, figsize=(10, 5)):
    """Display فقط، لا يؤثر على training ولا gradients ولا optimizer."""
    images = images.detach().cpu().float().clamp(0, 1)
    grid = make_grid(images, nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def save_epoch_visuals(model, loader, dataset_name: str, seed: int, epoch: int):
    
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)

    x = next(iter(loader))[0].to(DEVICE).float().clamp(0, 1)
    x = x[:16]
    recon = model.reconstruct(x, deterministic=True)
    samples = model.generate(16, DEVICE)

    recon_grid = torch.cat([x.detach().cpu(), recon.detach().cpu()], dim=0)
    samples = samples.detach().cpu()

    # نفس أسماء الملفات الأصلية حتى لا يتغير أي مسار في باقي النوتبوك.
    recon_path = FIGURE_DIR / f"{safe_name(dataset_name)}_seed{seed}_epoch{epoch:03d}_reconstruction.png"
    sample_path = FIGURE_DIR / f"{safe_name(dataset_name)}_seed{seed}_epoch{epoch:03d}_random_generation.png"

    # nrow=8 يعطي الشكل المطلوب: top originals, bottom reconstructions.
    save_image(recon_grid, str(recon_path), nrow=8)
    save_image(samples, str(sample_path), nrow=8)

    _display_tensor_grid(
        recon_grid,
        title=f"Epoch {epoch}: top originals, bottom reconstructions",
        nrow=8,
        figsize=(10, 5),
    )
    _display_tensor_grid(
        samples,
        title=f"Epoch {epoch}: random generated samples",
        nrow=8,
        figsize=(10, 5),
    )

    print("Saved visual:", recon_path)
    print("Saved samples:", sample_path)


In [ ]:
# ============================================================
# One-batch debug test
# ============================================================


def one_batch_debug(dataset_name: str, seed: int = 42):
    print(f"\nOne-batch debug: {dataset_name}, seed={seed}")
    set_seed(seed)
    train_loader, _, _ = build_dataloaders(dataset_name, seed)
    model = build_model(CFG)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scaler = make_grad_scaler(CFG.use_amp)

    batch = next(iter(train_loader))
    x = batch[0].to(DEVICE).float().clamp(0, 1)

    optimizer.zero_grad(set_to_none=True)
    with autocast_context(CFG.use_amp):
        recon_logits, mu, logvar, aux = model(x)
        loss, metrics = compute_loss(model, x, recon_logits, mu, logvar, aux, epoch=1, cfg=CFG)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
    scaler.step(optimizer)
    scaler.update()

    print("Debug metrics:", metrics)
    print("Output logits shape:", recon_logits.shape)
    return metrics

for ds_name in CFG.datasets:
    _ = one_batch_debug(ds_name, seed=42)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 5. Training

In [ ]:
# ============================================================
# Run training for MNIST and Fashion-MNIST
# ============================================================


def run_experiment(dataset_name: str, seed: int) -> Dict[str, Any]:
    set_seed(seed)
    paths = checkpoint_paths(dataset_name, seed)

    completed_epoch = 0
    if paths["latest"].exists():
        try:
            completed_epoch = int(load_checkpoint(paths["latest"]).get("epoch", 0))
        except Exception:
            completed_epoch = 0

    if (paths["best"].exists() and CFG.resume_if_available and not CFG.force_retrain
            and completed_epoch >= CFG.epochs):
        print(f"\nLoading existing checkpoint without retraining: {paths['best']}")
        model, checkpoint = build_model_from_checkpoint(paths["best"], DEVICE)
        train_loader, test_loader, classes = build_dataloaders(dataset_name, seed)
        history = checkpoint.get("history", [])
        return {
            "dataset_name": dataset_name,
            "seed": seed,
            "model": model,
            "checkpoint": checkpoint,
            "train_loader": train_loader,
            "test_loader": test_loader,
            "classes": classes,
            "best_path": paths["best"],
            "latest_path": paths["latest"],
            "history": history,
            "loaded_from_checkpoint": True,
        }

    print(f"\nTraining: {dataset_name}, seed={seed}")
    train_loader, test_loader, classes = build_dataloaders(dataset_name, seed)
    model = build_model(CFG)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))
    scaler = make_grad_scaler(CFG.use_amp)

    history: List[Dict[str, float]] = []
    best_val_loss = float("inf")

    start_epoch = 1
    if (paths["latest"].exists() and CFG.resume_if_available and not CFG.force_retrain
            and 0 < completed_epoch < CFG.epochs):
        ck = load_checkpoint(paths["latest"])
        model.load_state_dict(ck["model_state_dict"], strict=True)
        if ck.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(ck["optimizer_state_dict"])
        if ck.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ck["scheduler_state_dict"])
        if ck.get("scaler_state_dict") is not None and scaler is not None:
            try:
                scaler.load_state_dict(ck["scaler_state_dict"])
            except Exception:
                pass
        history = list(ck.get("history", []))
        best_val_loss = float(ck.get("best_val_loss", float("inf")))
        start_epoch = completed_epoch + 1
        print(f"Resuming from epoch {start_epoch}")

    start_time = time.time()

    for epoch in range(start_epoch, CFG.epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, epoch, CFG)
        val_metrics = validate_one_epoch(model, test_loader, epoch, CFG)
        scheduler.step()

        row = {
            "dataset": dataset_name,
            "seed": seed,
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)

        # The composite objective contains a KL term whose weight varies over the
        # warm-up, so total validation loss is not comparable across epochs and is
        # not used to select a checkpoint. The final-epoch weights are retained.
        current_val = val_metrics["loss"]
        best_val_loss = min(best_val_loss, current_val)
        if epoch == CFG.epochs:
            save_checkpoint(
                paths["best"], model, optimizer, scheduler, scaler,
                CFG, dataset_name, seed, epoch, history, best_val_loss
            )

        save_checkpoint(
            paths["latest"], model, optimizer, scheduler, scaler,
            CFG, dataset_name, seed, epoch, history, best_val_loss
        )

        if epoch % CFG.save_every_epochs == 0 or epoch == CFG.epochs:
            save_checkpoint(
                paths["latest"], model, optimizer, scheduler, scaler,
                CFG, dataset_name, seed, epoch, history, best_val_loss
            )

        if epoch % CFG.visual_every_epochs == 0 or epoch == 1 or epoch == CFG.epochs:
            save_epoch_visuals(model, test_loader, dataset_name, seed, epoch)

        print(
            f"[{dataset_name}][seed {seed}][{epoch:03d}/{CFG.epochs}] "
            f"train_loss={train_metrics['loss']:.4f} val_loss={val_metrics['loss']:.4f} "
            f"val_psnr={val_metrics['psnr']:.2f} score={val_metrics['score_mean']:.3f}"
        )

    # Save history CSV
    pd.DataFrame(history).to_csv(paths["history"], index=False)
    print("Saved history:", paths["history"])
    print(f"Finished {dataset_name}, seed={seed} in {(time.time() - start_time)/60:.2f} min")

    # Reload best checkpoint for testing/evaluation
    best_model, checkpoint = build_model_from_checkpoint(paths["best"], DEVICE)

    return {
        "dataset_name": dataset_name,
        "seed": seed,
        "model": best_model,
        "checkpoint": checkpoint,
        "train_loader": train_loader,
        "test_loader": test_loader,
        "classes": classes,
        "best_path": paths["best"],
        "latest_path": paths["latest"],
        "history": history,
        "loaded_from_checkpoint": False,
    }


experiments: Dict[Tuple[str, int], Dict[str, Any]] = {}

for dataset_name in CFG.datasets:
    for seed in CFG.seeds:
        experiments[(dataset_name, seed)] = run_experiment(dataset_name, seed)

print("\nExperiments ready:")
for key, exp in experiments.items():
    print(key, "checkpoint:", exp["best_path"], "loaded:", exp["loaded_from_checkpoint"])


## 6. Evaluation

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from typing import Dict, Any

def plot_history(exp: Dict[str, Any]):
    history = exp.get("history", [])
    if len(history) == 0:
        print("No history found for", exp["dataset_name"], exp["seed"])
        return

    df = pd.DataFrame(history)
    title_prefix = f"{exp['dataset_name']} | seed {exp['seed']}"

    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["train_loss"], label="train loss")
    plt.plot(df["epoch"], df["val_loss"], label="val loss")
    plt.title(title_prefix + " — Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    ax = plt.gca()
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["val_psnr"], label="val PSNR")
    plt.title(title_prefix + " — Validation PSNR")
    plt.xlabel("Epoch")
    plt.ylabel("PSNR")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

for exp in experiments.values():
    plot_history(exp)

In [ ]:
# ============================================================
# Load checkpoint manually without training
#============================================================


def load_experiment_without_training(dataset_name: str, seed: int) -> Dict[str, Any]:
    paths = checkpoint_paths(dataset_name, seed)
    model, checkpoint = build_model_from_checkpoint(paths["best"], DEVICE)
    train_loader, test_loader, classes = build_dataloaders(dataset_name, seed)
    history = checkpoint.get("history", [])

    exp = {
        "dataset_name": dataset_name,
        "seed": seed,
        "model": model,
        "checkpoint": checkpoint,
        "train_loader": train_loader,
        "test_loader": test_loader,
        "classes": classes,
        "best_path": paths["best"],
        "latest_path": paths["latest"],
        "history": history,
        "loaded_from_checkpoint": True,
    }
    print("Loaded:", paths["best"])
    return exp

#

In [ ]:
# ============================================================
# Test reconstruction for one input image from dataset
# ============================================================


def get_sample_from_test_loader(test_loader: DataLoader, sample_index: int = 0) -> Tuple[torch.Tensor, int]:
    """Fetch one sample by index from the test loader."""
    if sample_index < 0:
        raise ValueError("sample_index must be >= 0")

    seen = 0
    for batch in test_loader:
        x_batch, y_batch = batch[0], batch[1]
        batch_size = x_batch.size(0)
        if seen + batch_size > sample_index:
            local_idx = sample_index - seen
            return x_batch[local_idx:local_idx+1], int(y_batch[local_idx].item())
        seen += batch_size

    raise IndexError(f"sample_index {sample_index} is out of range.")


@torch.no_grad()
def test_reconstruction_from_dataset(exp: Dict[str, Any], sample_index: int = 0, deterministic: bool = True):
    """Run reconstruction test on one test-set image and show/save comparison."""
    model = exp["model"]
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)

    x_cpu, label = get_sample_from_test_loader(exp["test_loader"], sample_index)
    x = x_cpu.to(DEVICE).float().clamp(0, 1)

    recon = model.reconstruct(x, deterministic=deterministic)

    mse, psnr = mse_psnr(recon, x)
    ssim_value = 1.0 - float(simple_ssim_loss(recon, x).detach().cpu())

    class_name = exp["classes"][label] if label < len(exp["classes"]) else str(label)
    title = (
        f"{exp['dataset_name']} | seed={exp['seed']} | index={sample_index} | "
        f"label={class_name} | PSNR={psnr:.2f} | SSIM={ssim_value:.3f}"
    )

    save_path = FIGURE_DIR / f"{safe_name(exp['dataset_name'])}_seed{exp['seed']}_test_index{sample_index}.png"
    show_original_recon_diff(x.cpu()[0], recon.cpu()[0], title=title, save_path=save_path)

    return {
        "dataset": exp["dataset_name"],
        "seed": exp["seed"],
        "sample_index": sample_index,
        "label": label,
        "class_name": class_name,
        "mse": mse,
        "psnr": psnr,
        "ssim": ssim_value,
        "figure_path": str(save_path),
    }

# 
single_image_tests = []
for exp in experiments.values():
    single_image_tests.append(test_reconstruction_from_dataset(exp, sample_index=0))

pd.DataFrame(single_image_tests)


In [ ]:
# ============================================================
# Test a custom uploaded image
#  ============================================================


def preprocess_custom_image(image_path: str, invert: bool = False) -> torch.Tensor:
    """Load external image and convert it to [1, 1, 28, 28] tensor."""
    img = Image.open(image_path).convert("L").resize((CFG.image_size, CFG.image_size))
    arr = np.array(img).astype(np.float32) / 255.0
    if invert:
        arr = 1.0 - arr
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return tensor.clamp(0, 1)


@torch.no_grad()
def test_custom_image(exp: Dict[str, Any], image_path: str, invert: bool = False):
    """Test reconstruction on an uploaded/custom image."""
    model = exp["model"]
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)

    x_cpu = preprocess_custom_image(image_path, invert=invert)
    x = x_cpu.to(DEVICE).float()
    recon = model.reconstruct(x, deterministic=True)

    mse, psnr = mse_psnr(recon, x)
    ssim_value = 1.0 - float(simple_ssim_loss(recon, x).detach().cpu())

    title = f"Custom image with {exp['dataset_name']} model | PSNR={psnr:.2f} | SSIM={ssim_value:.3f}"
    save_path = FIGURE_DIR / f"custom_{safe_name(exp['dataset_name'])}_seed{exp['seed']}.png"
    show_original_recon_diff(x.cpu()[0], recon.cpu()[0], title=title, save_path=save_path)

    return {"mse": mse, "psnr": psnr, "ssim": ssim_value, "figure_path": str(save_path)}



In [ ]:
# ============================================================
# Random generation samples
# ============================================================

@torch.no_grad()
def show_random_generation(exp: Dict[str, Any], n: int = 16):
    model = exp["model"]
    model.eval()
    model.aqfe.set_depolarizing_noise(0.0)
    samples = model.generate(n, DEVICE)
    save_path = FIGURE_DIR / f"{safe_name(exp['dataset_name'])}_seed{exp['seed']}_random_generation_final.png"
    show_tensor_grid(samples, title=f"Random generated samples — {exp['dataset_name']} seed {exp['seed']}", nrow=8, save_path=save_path)

for exp in experiments.values():
    show_random_generation(exp, n=16)


In [ ]:
# ============================================================
# Install/import statistics metrics
#  ============================================================

import importlib.util
import subprocess

METRICS_READY = {
    "fid": False,
    "ssim_psnr": False,
    "lpips": False,
}

FrechetInceptionDistance = None
LearnedPerceptualImagePatchSimilarity = None
structural_similarity_index_measure = None
peak_signal_noise_ratio = None


def install_if_missing(import_name: str, pip_name: str):
    """Install a package only if its import name is missing."""
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


if CFG.run_statistics:
    try:
        install_if_missing("torchmetrics", "torchmetrics[image]")
        install_if_missing("torch_fidelity", "torch-fidelity")
        install_if_missing("lpips", "lpips")

        from torchmetrics.image.fid import FrechetInceptionDistance as _FID
        from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity as _LPIPS

        try:
            from torchmetrics.functional.image import (
                structural_similarity_index_measure as _SSIM,
                peak_signal_noise_ratio as _PSNR,
            )
        except Exception:
            from torchmetrics.functional.image.ssim import structural_similarity_index_measure as _SSIM
            from torchmetrics.functional.image.psnr import peak_signal_noise_ratio as _PSNR

        FrechetInceptionDistance = _FID
        LearnedPerceptualImagePatchSimilarity = _LPIPS
        structural_similarity_index_measure = _SSIM
        peak_signal_noise_ratio = _PSNR

        METRICS_READY = {"fid": True, "ssim_psnr": True, "lpips": True}
        print("Statistics metrics are ready:", METRICS_READY)

    except Exception as e:
        print("Warning: Some statistics packages could not be installed/imported.")
        print("The notebook will continue and unavailable metrics will be shown as NA.")
        print("Error:", repr(e))
else:
    print("CFG.run_statistics is False, so statistics metrics were skipped.")


In [ ]:
# ============================================================
# Evaluation helper functions
# ============================================================


def to_three_channels(x: torch.Tensor) -> torch.Tensor:
    if x.shape[1] == 1:
        return x.repeat(1, 3, 1, 1)
    return x


def prepare_for_fid(x: torch.Tensor, image_size: int) -> torch.Tensor:
    """FID expects uint8 images [B, 3, H, W] in [0, 255] when normalize=False."""
    x = to_three_channels(x.float().clamp(0, 1))
    x = F.interpolate(x, size=(image_size, image_size), mode="bilinear", align_corners=False)
    x = (x * 255.0).clamp(0, 255).to(torch.uint8)
    return x


def prepare_for_lpips(x: torch.Tensor, image_size: int) -> torch.Tensor:
    """LPIPS expects RGB-like images in [-1, 1]."""
    x = to_three_channels(x.float().clamp(0, 1))
    x = F.interpolate(x, size=(image_size, image_size), mode="bilinear", align_corners=False)
    return (x * 2.0 - 1.0).clamp(-1, 1)


def mean_std(values: List[float]) -> Tuple[float, float]:
    arr = np.array(values, dtype=np.float64)
    if len(arr) == 0:
        return float("nan"), float("nan")
    if len(arr) == 1:
        return float(arr.mean()), 0.0
    return float(arr.mean()), float(arr.std(ddof=1))


def fmt(mean: float, std: float, digits: int = 2) -> str:
    if np.isnan(mean):
        return "NA"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


In [ ]:
# ============================================================
# Collect reconstructions in ideal/noisy environment
# ============================================================

@torch.no_grad()
def collect_reconstruction_pairs(exp: Dict[str, Any], noisy: bool = False, noise_p: float = 0.0):
    model = exp["model"]
    loader = exp["test_loader"]
    model.eval()

    model.aqfe.set_depolarizing_noise(noise_p if noisy else 0.0)

    real_list = []
    fake_list = []

    for batch_idx, batch in enumerate(loader):
        if CFG.eval_max_batches is not None and batch_idx >= CFG.eval_max_batches:
            break

        x = batch[0].to(DEVICE, non_blocking=True).float().clamp(0, 1)
        recon = model.reconstruct(x, deterministic=True)

        real_list.append(x.detach().cpu())
        fake_list.append(recon.detach().cpu())

    model.aqfe.set_depolarizing_noise(0.0)

    real = torch.cat(real_list, dim=0)
    fake = torch.cat(fake_list, dim=0)

    return real, fake


In [ ]:
# ============================================================
# Compute FID / SSIM / PSNR / LPIPS
# ============================================================

@torch.no_grad()
def compute_ssim_psnr_lpips(real: torch.Tensor, fake: torch.Tensor) -> Tuple[List[float], List[float], List[float]]:
    ssim_values: List[float] = []
    psnr_values: List[float] = []
    lpips_values: List[float] = []

    if not METRICS_READY.get("ssim_psnr", False):
        print("SSIM/PSNR metrics are not available; returning NA.")
        return ssim_values, psnr_values, lpips_values

    lpips_metric = None
    if METRICS_READY.get("lpips", False) and LearnedPerceptualImagePatchSimilarity is not None:
        try:
            lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=False).to(DEVICE)
            lpips_metric.eval()
        except Exception as e:
            print("LPIPS unavailable during runtime; LPIPS will be NA. Error:", repr(e))
            lpips_metric = None

    n = real.shape[0]
    batch_size = CFG.eval_batch_size

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        x = real[start:end].to(DEVICE).float().clamp(0, 1)
        y = fake[start:end].to(DEVICE).float().clamp(0, 1)

        try:
            ssim_val = structural_similarity_index_measure(y, x, data_range=1.0)
            psnr_val = peak_signal_noise_ratio(y, x, data_range=1.0)
            ssim_values.append(float(ssim_val.detach().cpu()))
            psnr_values.append(float(psnr_val.detach().cpu()))
        except Exception as e:
            print("SSIM/PSNR batch failed; skipping this batch. Error:", repr(e))

        if lpips_metric is not None:
            try:
                x_lpips = prepare_for_lpips(x, CFG.lpips_image_size)
                y_lpips = prepare_for_lpips(y, CFG.lpips_image_size)
                lpips_val = lpips_metric(y_lpips, x_lpips)
                lpips_values.append(float(lpips_val.detach().cpu()))
            except Exception as e:
                print("LPIPS batch failed; skipping this batch. Error:", repr(e))

    return ssim_values, psnr_values, lpips_values


@torch.no_grad()
def compute_fid_values(real: torch.Tensor, fake: torch.Tensor) -> List[float]:
    """FID over the full set in a single pass.

    The estimator is biased upward at small sample sizes, so the whole set is
    accumulated into one metric object rather than averaged over batches.
    """
    if not METRICS_READY.get("fid", False) or FrechetInceptionDistance is None:
        print("FID metric is not available; returning NA.")
        return []

    n = real.shape[0]
    if n < 32:
        print("Warning: fewer than 32 images. FID may be unavailable.")
        return []

    metric = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
    with torch.no_grad():
        for start in range(0, n, 128):
            end = min(start + 128, n)
            metric.update(prepare_for_fid(real[start:end].to(DEVICE), CFG.fid_image_size),
                          real=True)
        for start in range(0, n, 128):
            end = min(start + 128, n)
            metric.update(prepare_for_fid(fake[start:end].to(DEVICE), CFG.fid_image_size),
                          real=False)
        value = float(metric.compute().detach().cpu())

    del metric
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return [value]


def evaluate_experiment_statistics(exp: Dict[str, Any], environment: str, noisy: bool) -> Dict[str, Any]:
    print(f"\nEvaluating {exp['dataset_name']} | seed {exp['seed']} | {environment}")

    real, fake = collect_reconstruction_pairs(
        exp,
        noisy=noisy,
        noise_p=CFG.noisy_depolarizing_p,
    )

    print("Collected:", real.shape[0], "images")

    ssim_values, psnr_values, lpips_values = compute_ssim_psnr_lpips(real, fake)
    fid_values = compute_fid_values(real, fake)

    fid_mean, fid_std = mean_std(fid_values)
    ssim_mean, ssim_std = mean_std(ssim_values)
    psnr_mean, psnr_std = mean_std(psnr_values)
    lpips_mean, lpips_std = mean_std(lpips_values)

    return {
        "Dataset": exp["dataset_name"],
        "Seed": exp["seed"],
        "Category": "Quantum",
        "Method": "AQFE-QCVAE",
        "Environment": environment,
        "Images": int(real.shape[0]),
        "FID_mean": fid_mean,
        "FID_std": fid_std,
        "SSIM_mean": ssim_mean,
        "SSIM_std": ssim_std,
        "PSNR_mean": psnr_mean,
        "PSNR_std": psnr_std,
        "LPIPS_mean": lpips_mean,
        "LPIPS_std": lpips_std,
        "FID": fmt(fid_mean, fid_std, digits=1),
        "SSIM": fmt(ssim_mean, ssim_std, digits=3),
        "PSNR": fmt(psnr_mean, psnr_std, digits=2),
        "LPIPS": fmt(lpips_mean, lpips_std, digits=3),
    }


In [ ]:
# ============================================================
# Run statistical analysis for all saved models
# ============================================================

statistics_rows = []

if CFG.run_statistics:
    for exp in experiments.values():
        statistics_rows.append(evaluate_experiment_statistics(exp, environment="Ideal", noisy=False))
        statistics_rows.append(evaluate_experiment_statistics(exp, environment="Noisy", noisy=True))

    statistics_df = pd.DataFrame(statistics_rows)

    stats_path = TABLE_DIR / "aqfe_qcvae_statistics_detailed.csv"
    statistics_df.to_csv(stats_path, index=False)

    print("Saved detailed statistics:", stats_path)
    display_cols = ["Dataset", "Seed", "Category", "Method", "Environment", "Images", "FID", "SSIM", "PSNR", "LPIPS"]
    display(statistics_df[display_cols])
else:
    statistics_df = pd.DataFrame()
    print("Statistics skipped because CFG.run_statistics is False.")


In [ ]:
# ============================================================
# Paper-style table
#  ============================================================


def aggregate_for_paper(statistics_df: pd.DataFrame) -> pd.DataFrame:
    if statistics_df is None or statistics_df.empty:
        return pd.DataFrame()

    rows = []

    for dataset_name in statistics_df["Dataset"].unique():
        ds_df = statistics_df[statistics_df["Dataset"] == dataset_name]
        row = {
            "Dataset": dataset_name,
            "Category": "Quantum",
            "Method": "AQFE-QCVAE",
        }

        for env in ["Ideal", "Noisy"]:
            env_df = ds_df[ds_df["Environment"] == env]
            if env_df.empty:
                continue

            for metric, digits in [("FID", 1), ("SSIM", 3), ("PSNR", 2), ("LPIPS", 3)]:
                seed_count = env_df["Seed"].nunique()

                if seed_count > 1:
                    # True mean ± std across different training seeds
                    values = env_df[f"{metric}_mean"].astype(float).values
                    values = [v for v in values if not np.isnan(v)]
                    m, s = mean_std(values)
                else:
                    # Single seed: use within-test chunks/batches std
                    m = float(env_df[f"{metric}_mean"].iloc[0])
                    s = float(env_df[f"{metric}_std"].iloc[0])

                row[f"{env} {metric}"] = fmt(m, s, digits=digits)

        rows.append(row)

    return pd.DataFrame(rows)

paper_table = aggregate_for_paper(statistics_df)

if not paper_table.empty:
    paper_path = TABLE_DIR / "aqfe_qcvae_paper_table.csv"
    paper_table.to_csv(paper_path, index=False)

    print("Saved paper table:", paper_path)
    display(paper_table)
else:
    print("No paper table generated because statistics_df is empty.")


In [ ]:
# ============================================================
# Recommended parameter presets
# ============================================================

presets = pd.DataFrame([
    {
        "Goal": "Debug only",
        "mode": "debug",
        "epochs": 1,
        "latent_dim": 32,
        "beta_max": 0.05,
        "notes": "Use this first to catch errors quickly."
    },
    {
        "Goal": "Best reconstruction",
        "mode": "paper",
        "epochs": 80,
        "latent_dim": 48,
        "beta_max": 0.03,
        "notes": "Sharper input-to-reconstruction results; generation prior may be weaker."
    },
    {
        "Goal": "Balanced VAE",
        "mode": "paper",
        "epochs": 80,
        "latent_dim": 32,
        "beta_max": 0.05,
        "notes": "Recommended default for MNIST and Fashion-MNIST."
    },
    {
        "Goal": "Better random generation",
        "mode": "paper",
        "epochs": 120,
        "latent_dim": 32,
        "beta_max": 0.10,
        "notes": "May reduce reconstruction sharpness but improves latent regularization."
    },
    {
        "Goal": "Paper mean ± std",
        "mode": "paper",
        "epochs": 80,
        "latent_dim": 32,
        "beta_max": 0.05,
        "notes": "Use seeds=(42,123,2026) for stronger statistical reporting."
    },
])

display(presets)


In [ ]:
# ============================================================
# Final checklist
# ===========================================================

print("Notebook completed.")
print("Checkpoints:", CHECKPOINT_DIR)
print("Figures:", FIGURE_DIR)
print("Tables:", TABLE_DIR)

if "experiments" in globals():
    print("Number of experiments:", len(experiments))
if "statistics_df" in globals() and isinstance(statistics_df, pd.DataFrame):
    print("Statistics rows:", len(statistics_df))
if "paper_table" in globals() and isinstance(paper_table, pd.DataFrame):
    print("Paper table rows:", len(paper_table))


In [ ]:
# ============================================================
# Ablation configuration
# ============================================================

import copy as _copy
from dataclasses import asdict as _asdict

AB_DIR = BASE_DIR / "ablation"
AB_TABLE_DIR = AB_DIR / "tables"
AB_FIG_DIR = AB_DIR / "figures"
AB_CKPT_DIR = AB_DIR / "checkpoints"
for _p in [AB_DIR, AB_TABLE_DIR, AB_FIG_DIR, AB_CKPT_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

AB_CFG = _copy.deepcopy(CFG)
AB_SEEDS = CFG.seeds
AB_CFG.epochs = 40                # ablation schedule; the main model uses CFG.epochs
AB_CFG.seeds = AB_SEEDS
AB_FORCE_RETRAIN = False

AB_ALL = ["CNN-VAE", "AQFE-D", "AQFE-DG", "AQFE-DQ",
          "QFR-VAE", "AQFE-DG-MLP", "AQFE-DG-MLPw", "beta-VAE"]
AB_VARIANTS = AB_ALL

AB_LABELS = {
    "CNN-VAE":      "Conv-VAE (raw image)",
    "AQFE-D":       "Conv-VAE + descriptors",
    "AQFE-DG":      "Conv-VAE + descriptors + gating",
    "AQFE-DQ":      "Conv-VAE + descriptors + qubit (no gating)",
    "QFR-VAE":      "QFR-VAE (proposed)",
    "AQFE-DG-MLP":  "Descriptors + gating + MLP (param-matched)",
    "AQFE-DG-MLPw": "Descriptors + gating + MLP (wide)",
    "beta-VAE":     "beta-VAE (raw image)",
}
AB_CATEGORY = {k: ("Quantum-inspired" if k in ("QFR-VAE", "AQFE-DQ") else "Classical")
               for k in AB_ALL}
AB_CFG_OVERRIDES = {"beta-VAE": {"beta_max": 1.0}}

def ab_cfg_for(v):
    c = _copy.deepcopy(AB_CFG)
    for k, val in AB_CFG_OVERRIDES.get(v, {}).items():
        setattr(c, k, val)
    return c

print("Variants :", AB_VARIANTS)
print("Seeds    :", AB_SEEDS)
print("Epochs   :", AB_CFG.epochs)
print("Datasets :", AB_CFG.datasets)
print("Runs     :", len(AB_VARIANTS) * len(AB_CFG.datasets) * len(AB_SEEDS))


In [ ]:
# ============================================================
# Front-end blocks and variant builder


class _NullQuantum(nn.Module):
    def __init__(self):
        super().__init__(); self.depolarizing_p = 0.0
        self.register_buffer("_zero", torch.zeros(()), persistent=False)
    def regularization_loss(self): return self._zero

class MLPReadout(nn.Module):
    def __init__(self, feature_dim=8, hidden=3, out_dim=4):
        super().__init__(); self.depolarizing_p = 0.0
        self.net = nn.Sequential(nn.Linear(feature_dim, hidden), nn.Tanh(),
                                 nn.Linear(hidden, out_dim), nn.Tanh())
    def forward(self, f):
        o = self.net(f.float())
        if self.depolarizing_p > 0: o = max(0.0, 1.0-float(self.depolarizing_p))*o
        return o
    def regularization_loss(self):
        return sum(p.pow(2).mean() for p in self.net.parameters())

class FlexibleAQFE(nn.Module):
    """Covers the descriptor-based variants via three switches."""
    def __init__(self, image_size, patch_size, stride, embed_dim,
                 use_gating=True, readout="qubit", q_layers=3, theta_scale=math.pi, mlp_hidden=3):
        super().__init__()
        self.image_size, self.embed_dim = image_size, embed_dim
        self.use_gating, self.readout_kind, self._noise_p = use_gating, readout, 0.0
        self.extractor = AQFEFeatureExtractor(patch_size=patch_size, stride=stride)
        fd = self.extractor.feature_dim
        if readout == "qubit":
            self.quantum = SingleQubitDataReuploading(fd, q_layers, theta_scale); extra = 4
        elif readout == "mlp":
            self.quantum = MLPReadout(fd, mlp_hidden, 4); extra = 4
        else:
            self.quantum = _NullQuantum(); extra = 0
        if use_gating:
            self.importance_gate = nn.Sequential(nn.Linear(fd,16), nn.ReLU(inplace=True),
                                                 nn.Linear(16,1), nn.Sigmoid())
        self.feature_projector = nn.Sequential(nn.Linear(fd+extra,32), nn.ReLU(inplace=True),
                                               nn.Linear(32,embed_dim), nn.ReLU(inplace=True))
    def extract_handcrafted_features(self, x): return self.extractor(x)
    def set_depolarizing_noise(self, p):
        self._noise_p = float(p)
        if hasattr(self.quantum, "depolarizing_p"): self.quantum.depolarizing_p = float(p)
    def forward(self, x):
        f = self.extractor(x)
        if self.use_gating:
            s = self.importance_gate(f); f = f*(0.5+s)
            aux = {"score_mean": s.mean(), "score_std": s.std(unbiased=False)}
        else:
            z = f.new_zeros(()); aux = {"score_mean": z, "score_std": z}
        if self.readout_kind == "none":
            h = f*(max(0.0,1.0-self._noise_p) if self._noise_p > 0 else 1.0)
        else:
            h = torch.cat([f, self.quantum(f)], dim=-1)
        e = self.feature_projector(h); b = x.shape[0]
        return e.transpose(1,2).contiguous().view(b, self.embed_dim, self.image_size, self.image_size), aux

class PlainConvStem(nn.Module):
    """No descriptors, no gating, no readout: a learned convolutional stem."""
    def __init__(self, image_size, patch_size, stride, embed_dim):
        super().__init__()
        self.image_size, self.embed_dim, self._noise_p = image_size, embed_dim, 0.0
        self.extractor = AQFEFeatureExtractor(patch_size=patch_size, stride=stride)
        self.quantum = _NullQuantum()
        self.stem = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=patch_size,stride=1,padding=patch_size//2), nn.ReLU(inplace=True),
            nn.Conv2d(32,embed_dim,kernel_size=1), nn.ReLU(inplace=True))
    def extract_handcrafted_features(self, x): return self.extractor(x)
    def set_depolarizing_noise(self, p): self._noise_p = float(p)
    def forward(self, x):
        f = self.stem(x.float().clamp(0,1))
        if self._noise_p > 0: f = max(0.0,1.0-self._noise_p)*f
        z = f.new_zeros(()); return f, {"score_mean": z, "score_std": z}

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

AB_QPARAMS = count_params(SingleQubitDataReuploading(8, AB_CFG.q_layers, AB_CFG.theta_scale))
AB_H_MATCHED = min(range(1,65), key=lambda h: abs(count_params(MLPReadout(8,h,4)) - AB_QPARAMS))
AB_H_WIDE = 48

AB_SPEC = {
    "CNN-VAE": dict(kind="stem"), "beta-VAE": dict(kind="stem"),
    "AQFE-D":  dict(kind="flex", use_gating=False, readout="none"),
    "AQFE-DG": dict(kind="flex", use_gating=True,  readout="none"),
    "AQFE-DQ": dict(kind="flex", use_gating=False, readout="qubit"),
    "QFR-VAE": dict(kind="original"),
    "AQFE-DG-MLP":  dict(kind="flex", use_gating=True, readout="mlp", mlp_hidden=AB_H_MATCHED),
    "AQFE-DG-MLPw": dict(kind="flex", use_gating=True, readout="mlp", mlp_hidden=AB_H_WIDE)}

def build_variant_model(variant, cfg):
    """Build the full model and swap ONLY the .aqfe front-end.
    QFR-VAE keeps the original AQFEModule untouched."""
    spec = dict(AB_SPEC[variant]); kind = spec.pop("kind")
    model = AQFEQCVAE(**model_kwargs_from_cfg(cfg))
    if kind == "original":
        return model.to(DEVICE)
    common = dict(image_size=cfg.image_size, patch_size=cfg.patch_size,
                  stride=cfg.stride, embed_dim=cfg.aqfe_embed_dim)
    model.aqfe = (PlainConvStem(**common) if kind == "stem"
                  else FlexibleAQFE(q_layers=cfg.q_layers, theta_scale=cfg.theta_scale,
                                    **common, **spec))
    return model.to(DEVICE)

print(f"Qubit readout block : {AB_QPARAMS} params")
print(f"Matched MLP         : hidden={AB_H_MATCHED} -> {count_params(MLPReadout(8,AB_H_MATCHED,4))} params")
print(f"Wide MLP            : hidden={AB_H_WIDE} -> {count_params(MLPReadout(8,AB_H_WIDE,4))} params")

set_seed(42)
_x = torch.rand(4,1,AB_CFG.image_size,AB_CFG.image_size, device=DEVICE)
_r = []
for _v in AB_VARIANTS:
    _c = ab_cfg_for(_v); _m2 = build_variant_model(_v, _c); _m2.eval()
    with torch.no_grad(): _o,_,_,_ = _m2(_x)
    _r.append({"Variant": AB_LABELS[_v], "Category": AB_CATEGORY[_v],
               "Readout params": count_params(_m2.aqfe.quantum),
               "Front-end params": count_params(_m2.aqfe),
               "Total params": count_params(_m2), "beta_max": _c.beta_max,
               "Output": tuple(_o.shape)})
    del _m2
if DEVICE.type == "cuda": torch.cuda.empty_cache()
ab_param_table = pd.DataFrame(_r)
ab_param_table.to_csv(RR_TABLE_DIR / "ablation_parameter_counts.csv", index=False)
display(ab_param_table)


In [ ]:
# ============================================================
# Ablation training runner
# ============================================================

def ab_ckpt(variant, ds, seed, kind="best"):
    tag = "final".replace("_", "")
    return RR_CKPT_DIR / (f"{safe_name(variant)}_{safe_name(ds)}_{AB_CFG.mode}"
                          f"_e{AB_CFG.epochs}_sel-{tag}_seed{seed}_{kind}.pt")

def ab_load(variant, path, cfg):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = build_variant_model(variant, cfg)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    m.eval(); m.aqfe.set_depolarizing_noise(0.0)
    return m, ck

def run_variant(variant, ds, seed):
    cfg = ab_cfg_for(variant)
    p_best, p_final = ab_ckpt(variant, ds, seed, "best"), ab_ckpt(variant, ds, seed, "final")
    set_seed(seed)
    train_loader, test_loader, classes = build_dataloaders(ds, seed)

    if p_best.exists() and not AB_FORCE_RETRAIN:
        m, ck = ab_load(variant, p_best, cfg)
        print(f"  cached: {variant} | {ds} | seed {seed} (selected epoch {ck.get('epoch')})")
        return {"variant":variant,"dataset_name":ds,"seed":seed,"model":m,
                "train_loader":train_loader,"test_loader":test_loader,"classes":classes,
                "history":ck.get("history",[]),"best_path":p_best,"final_path":p_final,
                "selected_epoch":ck.get("epoch"),"cfg":cfg}

    set_seed(seed)
    model = build_variant_model(variant, cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1,cfg.epochs))
    scaler = make_grad_scaler(cfg.use_amp)
    history, best_score, best_epoch = [], -float("inf"), None
    t0 = time.time()

    for ep in range(1, cfg.epochs+1):
        tr = train_one_epoch(model, train_loader, opt, scaler, ep, cfg)
        va = validate_one_epoch(model, test_loader, ep, cfg)
        sch.step()
        history.append({"variant":variant,"dataset":ds,"seed":seed,"epoch":ep,
                        **{f"train_{k}":v for k,v in tr.items()},
                        **{f"val_{k}":v for k,v in va.items()}})

        score = selection_value(va, ep, cfg)
        if score is not None and score > best_score:
            best_score, best_epoch = score, ep
            torch.save({"variant":variant,"dataset_name":ds,"seed":seed,"epoch":ep,
                        "model_kwargs":model_kwargs_from_cfg(cfg),
                        "model_state_dict":model.state_dict(),"cfg":_asdict(cfg),
                        "history":history,"selection":"final",
                        "selection_score":best_score}, p_best)

        if ep == cfg.epochs:
            torch.save({"variant":variant,"dataset_name":ds,"seed":seed,"epoch":ep,
                        "model_kwargs":model_kwargs_from_cfg(cfg),
                        "model_state_dict":model.state_dict(),"cfg":_asdict(cfg),
                        "history":history,"selection":"final"}, p_final)

        if ep % 20 == 0 or ep == 1 or ep == cfg.epochs:
            print(f"    [{ep:03d}/{cfg.epochs}] train={tr['loss']:.4f} val={va['loss']:.4f} "
                  f"psnr={va['psnr']:.2f}")

    pd.DataFrame(history).to_csv(
        RR_TABLE_DIR / f"history_{safe_name(variant)}_{safe_name(ds)}_s{seed}.csv", index=False)
    print(f"  done in {(time.time()-t0)/60:.1f} min | selected epoch {best_epoch} "
          f"by {"final"}")

    if not p_best.exists():          # e.g. "final" == "final"
        p_best = p_final
    model, ck = ab_load(variant, p_best, cfg)
    return {"variant":variant,"dataset_name":ds,"seed":seed,"model":model,
            "train_loader":train_loader,"test_loader":test_loader,"classes":classes,
            "history":history,"best_path":p_best,"final_path":p_final,
            "selected_epoch":best_epoch,"cfg":cfg}


In [ ]:
# ============================================================
# Train every ablation configuration
# ============================================================

ab_experiments = {}
for _ds in AB_CFG.datasets:
    for _sd in AB_SEEDS:
        for _v in AB_VARIANTS:
            print(f"\n--- {_v} | {_ds} | seed {_sd} ---")
            ab_experiments[(_v, _ds, _sd)] = run_variant(_v, _ds, _sd)
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

print("\nConfigurations trained:", len(ab_experiments))


### Ablation evaluation

In [ ]:
# ============================================================
# Evaluate every variant, ideal and noisy
# ============================================================

ab_rows, ab_perbatch = [], {}
if CFG.run_statistics:
    for (v, ds, sd), e in ab_experiments.items():
        for env, noisy, p in [("Ideal", False, 0.0),
                              ("Noisy p=0.03", True, 0.03),
                              ("Noisy p=0.10", True, 0.10)]:
            print(f"eval {v} | {ds} | seed {sd} | {env}")
            real, fake = collect_reconstruction_pairs(e, noisy=noisy, noise_p=p)
            s, ps, lp = compute_ssim_psnr_lpips(real, fake)
            fid_v = fid_single_pass(real, fake)
            ab_perbatch[(v,ds,sd,env)] = {"SSIM":s,"PSNR":ps,"LPIPS":lp}
            ab_rows.append({"Variant":v,"Label":AB_LABELS[v],"Category":AB_CATEGORY[v],
                            "Dataset":ds,"Seed":sd,"Environment":env,
                            "Images":int(real.shape[0]),
                            "FID_mean":fid_v,"FID_std":float("nan"),
                            "SSIM_mean":mean_std(s)[0],"SSIM_std":mean_std(s)[1],
                            "PSNR_mean":mean_std(ps)[0],"PSNR_std":mean_std(ps)[1],
                            "LPIPS_mean":mean_std(lp)[0],"LPIPS_std":mean_std(lp)[1]})
            del real, fake
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
    ab_stats_df = pd.DataFrame(ab_rows)
    ab_stats_df.to_csv(AB_TABLE_DIR / "ablation_statistics_detailed.csv", index=False)
    display(ab_stats_df)
else:
    ab_stats_df = pd.DataFrame()


### Ablation summary table

In [ ]:
# ============================================================

# ============================================================

def ab_aggregate(df):
    if df is None or df.empty: return pd.DataFrame()
    order = {v:i for i,v in enumerate(AB_ALL)}
    out = []
    for ds in df["Dataset"].unique():
        for v in sorted(df["Variant"].unique(), key=lambda z: order.get(z,99)):
            sub = df[(df["Dataset"]==ds)&(df["Variant"]==v)]
            if sub.empty: continue
            row = {"Dataset":ds,"Category":AB_CATEGORY[v],"Method":AB_LABELS[v],
                   "Runs":sub["Seed"].nunique()}
            for env in sub["Environment"].unique():
                e = sub[sub["Environment"]==env]
                for metric, dg in [("FID",1),("SSIM",4),("PSNR",2),("LPIPS",4)]:
                    vals = [x for x in e[f"{metric}_mean"].astype(float).values if not np.isnan(x)]
                    if len(vals) > 1: m, s = mean_std(vals)
                    elif len(vals) == 1: m, s = float(vals[0]), 0.0
                    else: m, s = float("nan"), float("nan")
                    row[f"{env} {metric}"] = fmt(m, s, digits=dg)
            out.append(row)
    return pd.DataFrame(out)

ab_table = ab_aggregate(ab_stats_df)
if not ab_table.empty:
    ab_table.to_csv(AB_TABLE_DIR / "ablation_paper_table.csv", index=False)
    display(ab_table)


### Paired comparisons

In [ ]:
# ============================================================
# Paired significance tests  
# ============================================================

try:
    from scipy import stats as _st; _SCIPY = True
except Exception:
    _SCIPY = False; print("scipy unavailable; reporting mean differences only.")

def cliffs_delta(a,b):
    a,b = np.asarray(a,float), np.asarray(b,float)
    if len(a)==0 or len(b)==0: return float("nan")
    gt = sum((x>b).sum() for x in a); lt = sum((x<b).sum() for x in a)
    return float(gt-lt)/(len(a)*len(b))

def paired_compare(va, vb, label):
    rows = []
    envs = sorted({k[3] for k in ab_perbatch})
    for ds in AB_CFG.datasets:
        for env in envs:
            for metric in ["SSIM","PSNR","LPIPS"]:
                A,B = [],[]
                for sd in AB_SEEDS:
                    ka,kb = ab_perbatch.get((va,ds,sd,env)), ab_perbatch.get((vb,ds,sd,env))
                    if not ka or not kb: continue
                    n = min(len(ka[metric]), len(kb[metric]))
                    A += ka[metric][:n]; B += kb[metric][:n]
                if len(A) < 3: continue
                d = np.array(A)-np.array(B)
                pw = pt = float("nan")
                if _SCIPY and np.any(d != 0):
                    try:
                        pw = float(_st.wilcoxon(A,B).pvalue); pt = float(_st.ttest_rel(A,B).pvalue)
                    except Exception: pass
                better = (d.mean()>0) if metric in ("SSIM","PSNR") else (d.mean()<0)
                rows.append({"Comparison":label,"Dataset":ds,"Environment":env,"Metric":metric,
                             "n":len(A), va:round(float(np.mean(A)),5), vb:round(float(np.mean(B)),5),
                             "Diff":round(float(d.mean()),5),"Wilcoxon p":pw,"t p":pt,
                             "Cliff d":round(cliffs_delta(A,B),3),
                             "Sig":(pw<0.05) if not np.isnan(pw) else None,
                             "Favours first":better})
    return pd.DataFrame(rows)

rr_tests = {}
plan = [("QFR-VAE","AQFE-DG-MLP","qubit block vs parameter-matched MLP"),
        ("QFR-VAE","AQFE-DG-MLPw","capacity control: qubit block vs wide MLP"),
        ("QFR-VAE","AQFE-DG","does the readout block add anything?"),
        ("QFR-VAE","AQFE-DQ","does gating add anything?"),
        ("AQFE-DG","AQFE-D","does gating add anything, without the circuit?"),
        ("AQFE-D","CNN-VAE","handcrafted descriptors vs a learned conv stem"),
        ("QFR-VAE","CNN-VAE","full model vs classical baseline")]
for a,b,lab in plan:
    if a not in AB_VARIANTS or b not in AB_VARIANTS: continue
    print("\\n" + "="*78); print(lab); print("="*78)
    t = paired_compare(a,b,lab); rr_tests[lab] = t; display(t)
if rr_tests:
    pd.concat(rr_tests.values()).to_csv(AB_TABLE_DIR / "ablation_significance.csv", index=False)


### Noise-robustness sweep

In [ ]:
# ============================================================
# Noise sweep and qualitative grid
# ============================================================

AB_NOISE_GRID = [0.0, 0.03, 0.10, 0.20, 0.30, 0.50]
_cv = [v for v in ["QFR-VAE","AQFE-DG-MLP","AQFE-DG","CNN-VAE"] if v in AB_VARIANTS]

curve = []
if CFG.run_statistics and ab_experiments:
    for ds in AB_CFG.datasets:
        for v in _cv:
            for p in AB_NOISE_GRID:
                S,P = [],[]
                for sd in AB_SEEDS:
                    e = ab_experiments.get((v,ds,sd))
                    if e is None: continue
                    real,fake = collect_reconstruction_pairs(e, noisy=(p>0), noise_p=p)
                    s,ps,_ = compute_ssim_psnr_lpips(real,fake); S += s; P += ps
                    del real,fake
                if not S: continue
                curve.append({"Dataset":ds,"Variant":v,"p":p,
                              "SSIM_mean":mean_std(S)[0],"SSIM_std":mean_std(S)[1],
                              "PSNR_mean":mean_std(P)[0],"PSNR_std":mean_std(P)[1]})
            if DEVICE.type=="cuda": torch.cuda.empty_cache()
    ab_curve_df = pd.DataFrame(curve)
    ab_curve_df.to_csv(AB_TABLE_DIR / "noise_sweep.csv", index=False)
    display(ab_curve_df)
    for ds in ab_curve_df["Dataset"].unique():
        sub = ab_curve_df[ab_curve_df["Dataset"]==ds]
        plt.figure(figsize=(6.2,4))
        for v in sub["Variant"].unique():
            s = sub[sub["Variant"]==v].sort_values("p")
            plt.errorbar(s["p"], s["SSIM_mean"], yerr=s["SSIM_std"], marker="o",
                         capsize=3, label=AB_LABELS[v])
        plt.xlabel("Depolarizing probability $p$"); plt.ylabel("SSIM")
        plt.title(f"{ds} - robustness to depolarizing noise")
        plt.grid(True, alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
        plt.savefig(AB_FIG_DIR / f"noise_sweep_{safe_name(ds)}.png", dpi=300); plt.show()

if ab_experiments:
    for ds in AB_CFG.datasets:
        sd = AB_SEEDS[0]
        base = ab_experiments.get(("QFR-VAE",ds,sd))
        if base is None: continue
        x = next(iter(base["test_loader"]))[0][:8].to(DEVICE).float().clamp(0,1)
        imgs, names = [x.detach().cpu()], ["Input"]
        for v in AB_VARIANTS:
            e = ab_experiments.get((v,ds,sd))
            if e is None: continue
            e["model"].aqfe.set_depolarizing_noise(0.0)
            with torch.no_grad():
                imgs.append(e["model"].reconstruct(x, deterministic=True).detach().cpu())
            names.append(AB_LABELS[v])
        fig, axes = plt.subplots(len(imgs), 8, figsize=(9, 1.15*len(imgs)))
        for i,(row,nm) in enumerate(zip(imgs,names)):
            for j in range(8):
                ax = axes[i,j]; ax.imshow(row[j,0], cmap="gray", vmin=0, vmax=1)
                ax.set_xticks([]); ax.set_yticks([])
                if j == 0: ax.set_ylabel(nm, rotation=0, ha="right", va="center", fontsize=6.5)
        plt.suptitle(f"{ds} - ablation reconstructions (seed {sd}, ideal)", fontsize=10)
        plt.tight_layout()
        plt.savefig(AB_FIG_DIR / f"ablation_qualitative_{safe_name(ds)}.png",
                    dpi=300, bbox_inches="tight"); plt.show()
